In [64]:
import pandas as pd
import numpy as np

In [65]:
# ĐỌC FILE KHẢO SÁT
FILE_XLSX = "data_2.xlsx"
df_raw = pd.read_excel(FILE_XLSX, sheet_name=0)

# Chuẩn hóa tên cột
df = df_raw.copy()
df.columns = [str(c).strip() for c in df.columns]

# Lọc các cột thang đo (đúng cấu trúc RE, EMP, DOC, SAT)
prefix = ("RE", "EMP", "DOC", "SAT")
item_cols = [c for c in df.columns if any(str(c).startswith(p) for p in prefix)]

# Ép kiểu số (Likert)
df[item_cols] = df[item_cols].apply(pd.to_numeric, errors="coerce")

# Loại dòng thiếu dữ liệu
df_items = df[item_cols].dropna().reset_index(drop=True)

print("Số mẫu hợp lệ:", len(df_items))

Số mẫu hợp lệ: 66


In [70]:
# ============================================================
# PHẦN CRONBACH'S ALPHA (CHUẨN KHÓA LUẬN) – thêm đầy đủ:
# 1) Alpha tổng
# 2) Corrected Item–Total Correlation
# 3) Alpha if item deleted
# 4) Gợi ý loại biến theo ngưỡng (ITC < 0.30)
# ============================================================

import numpy as np
import pandas as pd

def cronbach_alpha(items: pd.DataFrame) -> float:
    """Cronbach's alpha (bản an toàn)."""
    items = items.dropna()
    k = items.shape[1]
    if k < 2:
        return np.nan
    variances = items.var(ddof=1)
    total = items.sum(axis=1)
    total_var = total.var(ddof=1)
    if total_var == 0 or np.isnan(total_var):
        return np.nan
    return (k / (k - 1)) * (1 - variances.sum() / total_var)

def corrected_item_total_corr(items: pd.DataFrame) -> pd.Series:
    """Corrected item-total correlation: corr(item, total - item)."""
    items = items.dropna()
    if items.shape[1] < 2:
        return pd.Series(dtype=float)
    total = items.sum(axis=1)
    return pd.Series({c: items[c].corr(total - items[c]) for c in items.columns})

def alpha_if_item_deleted(items: pd.DataFrame) -> pd.Series:
    """Cronbach's alpha if delete each item (giống SPSS)."""
    items = items.dropna()
    out = {}
    for c in items.columns:
        out[c] = cronbach_alpha(items.drop(columns=[c]))
    return pd.Series(out)

def cronbach_report(items: pd.DataFrame, scale_name: str, itc_threshold: float = 0.30) -> pd.DataFrame:
    """
    Trả về bảng giống SPSS:
    - Corrected Item-Total Correlation
    - Cronbach's Alpha if Item Deleted
    + in thêm alpha tổng ở header (bạn có thể print riêng)
    """
    items = items.dropna()
    alpha_total = cronbach_alpha(items)
    itc = corrected_item_total_corr(items)
    alpha_del = alpha_if_item_deleted(items)

    rep = pd.DataFrame({
        "Corrected_Item_Total_Corr": itc,
        "Alpha_if_Item_Deleted": alpha_del
    }).sort_values("Corrected_Item_Total_Corr")

    rep["Flag_ITC_lt_0.30"] = rep["Corrected_Item_Total_Corr"] < itc_threshold

    print(f"\n===== {scale_name} =====")
    print(f"Cronbach's Alpha (overall) = {alpha_total:.3f} | k = {items.shape[1]} | n = {items.shape[0]}")
    print("Gợi ý loại biến: Flag_ITC_lt_0.30 = True (nếu có)")

    return rep

# ============================================================
# CÁCH DÙNG (sau khi bạn đã có df_items với cột RE1.., EMP1.., DOC1.., SAT1..)
# ============================================================

def get_block(df_items: pd.DataFrame, prefix: str) -> pd.DataFrame:
    cols = [c for c in df_items.columns if str(c).startswith(prefix)]
    if len(cols) < 2:
        raise ValueError(f"Thang đo {prefix} có <2 biến quan sát hoặc không tìm thấy cột.")
    return df_items[cols]

RE_block  = get_block(df_items, "RE")
EMP_block = get_block(df_items, "EMP")
DOC_block = get_block(df_items, "DOC")
SAT_block = get_block(df_items, "SAT")

RE_rep  = cronbach_report(RE_block,  "RE")
EMP_rep = cronbach_report(EMP_block, "EMP")
DOC_rep = cronbach_report(DOC_block, "DOC")
SAT_rep = cronbach_report(SAT_block, "SAT")

# Xem bảng kết quả (giống SPSS "Item-Total Statistics")
RE_rep



===== RE =====
Cronbach's Alpha (overall) = 0.882 | k = 5 | n = 66
Gợi ý loại biến: Flag_ITC_lt_0.30 = True (nếu có)

===== EMP =====
Cronbach's Alpha (overall) = 0.919 | k = 5 | n = 66
Gợi ý loại biến: Flag_ITC_lt_0.30 = True (nếu có)

===== DOC =====
Cronbach's Alpha (overall) = 0.940 | k = 5 | n = 66
Gợi ý loại biến: Flag_ITC_lt_0.30 = True (nếu có)

===== SAT =====
Cronbach's Alpha (overall) = 0.972 | k = 5 | n = 66
Gợi ý loại biến: Flag_ITC_lt_0.30 = True (nếu có)


,Corrected_Item_Total_Corr,Alpha_if_Item_Deleted,Flag_ITC_lt_0.30
RE3,0.578361,0.893435,False
RE5,0.637470,0.875251,False
RE4,0.786490,0.844117,False
RE1,0.811075,0.833516,False
RE2,0.818399,0.835507,False


In [67]:
# ============================================================
# EFA (1 KHUNG CODE) -> TRẢ KẾT QUẢ NGAY:
# - KMO (overall + per-item)
# - Bartlett (Chi-square, df, p-value)
# - Eigenvalues (Kaiser > 1)
# - Factor Loadings (PCA-based + Varimax)  <-- giống thực hành SPSS: PCA + Varimax
# - Communalities
#
# INPUT: file data.xlsx (cột câu hỏi dài tiếng Việt)
# Output: in kết quả trực tiếp
# ============================================================

import pandas as pd, numpy as np, re
from scipy import stats
from sklearn.decomposition import PCA

FILE_XLSX = "data_2.xlsx"
N_FACTORS = 4          # chạy theo mô hình lý thuyết RE-EMP-DOC-SAT (đổi 3 nếu theo Kaiser)
MIN_LOADING = 0.50     # ngưỡng loading để đánh dấu biến yếu

# -------------------------
# 0) READ + AUTO RENAME (B1/B2/B3/PHẦN C -> RE/EMP/DOC/SAT)
# -------------------------
df = pd.read_excel(FILE_XLSX, sheet_name=0)
df.columns = [str(c).strip() for c in df.columns]

def pick_cols(pattern: str):
    pat = re.compile(pattern, flags=re.IGNORECASE)
    return [c for c in df.columns if pat.search(str(c))]

b1_cols  = pick_cols(r"\bB1\b")
b2_cols  = pick_cols(r"\bB2\b")
b3_cols  = pick_cols(r"\bB3\b")
sat_cols = pick_cols(r"PHẦN\s*C")

rename_map = {}
for i, c in enumerate(b1_cols,  start=1): rename_map[c] = f"RE{i}"
for i, c in enumerate(b2_cols,  start=1): rename_map[c] = f"EMP{i}"
for i, c in enumerate(b3_cols,  start=1): rename_map[c] = f"DOC{i}"
for i, c in enumerate(sat_cols, start=1): rename_map[c] = f"SAT{i}"

df2 = df.rename(columns=rename_map)
item_cols = list(rename_map.values())

df_items = df2[item_cols].apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)

X = df_items.values
n, p = X.shape
print("EFA data shape (n, p) =", (n, p))
print("Items:", df_items.columns.tolist())

# Standardize => PCA trên ma trận tương quan (giống SPSS)
Xz = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)

# -------------------------
# 1) KMO & Bartlett
# -------------------------
def corr_matrix(xz: np.ndarray) -> np.ndarray:
    return np.corrcoef(xz, rowvar=False)

def kmo_test(R: np.ndarray):
    invR = np.linalg.inv(R)
    D = np.diag(1 / np.sqrt(np.diag(invR)))
    partial = -D @ invR @ D
    np.fill_diagonal(partial, 0.0)

    R2 = R.copy()
    np.fill_diagonal(R2, 0.0)

    r2 = (R2**2).sum()
    p2 = (partial**2).sum()
    kmo_overall = r2 / (r2 + p2)

    r2_j = (R2**2).sum(axis=0)
    p2_j = (partial**2).sum(axis=0)
    kmo_j = r2_j / (r2_j + p2_j)
    return kmo_overall, kmo_j

def bartlett_sphericity(R: np.ndarray, n_samples: int):
    detR = np.linalg.det(R)
    detR = max(detR, 1e-30)  # chống lỗi số
    chi2 = -(n_samples - 1 - (2*p + 5)/6) * np.log(detR)
    df_ = p*(p-1)/2
    pval = 1 - stats.chi2.cdf(chi2, df_)
    return chi2, df_, pval

R = corr_matrix(Xz)
kmo_overall, kmo_per_item = kmo_test(R)
chi2, df_b, p_b = bartlett_sphericity(R, n)

print("\n================= KMO & BARTLETT =================")
print(f"KMO (overall) = {kmo_overall:.3f}")
print(f"Bartlett: chi2={chi2:.3f}, df={df_b:.0f}, p-value={p_b:.6g}")

kmo_item_df = pd.DataFrame({"Item": df_items.columns, "KMO_item": kmo_per_item}).sort_values("KMO_item")
print("\nKMO per item (sorted):")
print(kmo_item_df.to_string(index=False))

# -------------------------
# 2) Eigenvalues (Kaiser > 1)
# -------------------------
pca_all = PCA(n_components=p).fit(Xz)
eigenvalues = pca_all.explained_variance_
eigen_table = pd.DataFrame({"Factor": np.arange(1, p+1), "Eigenvalue": eigenvalues})
n_kaiser = int((eigenvalues > 1).sum())

print("\n================= EIGENVALUES (Kaiser) =================")
print(eigen_table.round(4).to_string(index=False))
print(f"\nGợi ý số nhân tố theo Eigenvalue>1: {n_kaiser}")

# -------------------------
# 3) Varimax rotation + Factor loadings
# -------------------------
def varimax(Phi, gamma=1.0, q=50, tol=1e-6):
    p_, k_ = Phi.shape
    Rm = np.eye(k_)
    d = 0
    for _ in range(q):
        d_old = d
        Lambda = Phi @ Rm
        u, s, vh = np.linalg.svd(
            Phi.T @ (Lambda**3 - (gamma/p_) * Lambda @ np.diag(np.diag(Lambda.T @ Lambda)))
        )
        Rm = u @ vh
        d = s.sum()
        if d_old != 0 and d/d_old < 1 + tol:
            break
    return Phi @ Rm

# PCA loadings (unrotated) cho dữ liệu chuẩn hóa: components_.T * sqrt(eigenvalues)
pca = PCA(n_components=N_FACTORS).fit(Xz)
unrot_loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
rot_loadings = varimax(unrot_loadings)

loadings_df = pd.DataFrame(rot_loadings, index=df_items.columns, columns=[f"F{i+1}" for i in range(N_FACTORS)])

print("\n================= FACTOR LOADINGS (Varimax) =================")
print(loadings_df.round(3).to_string())

# Communalities
communalities = (rot_loadings**2).sum(axis=1)
comm_df = pd.DataFrame({"Item": df_items.columns, "Communality": communalities}).sort_values("Communality")

print("\n================= COMMUNALITIES =================")
print(comm_df.round(3).to_string(index=False))

# -------------------------
# 4) (Tuỳ chọn) đánh dấu biến yếu / tải chéo
# -------------------------
absL = np.abs(rot_loadings)
top1 = absL.max(axis=1)
top2 = np.partition(absL, -2, axis=1)[:, -2] if N_FACTORS >= 2 else np.zeros_like(top1)
gap = top1 - top2

quality = pd.DataFrame({
    "Item": df_items.columns,
    "Top1_loading": top1,
    "Top2_loading": top2,
    "Gap": gap,
    f"Weak_loading(<{MIN_LOADING})": top1 < MIN_LOADING,
    "Cross_loading(Gap<0.20)": gap < 0.20
}).sort_values([f"Weak_loading(<{MIN_LOADING})", "Cross_loading(Gap<0.20)", "Top1_loading"],
               ascending=[False, False, True])

print("\n================= QUALITY CHECK =================")
print(quality.round(3).to_string(index=False))


EFA data shape (n, p) = (66, 20)
Items: ['RE1', 'RE2', 'RE3', 'RE4', 'RE5', 'EMP1', 'EMP2', 'EMP3', 'EMP4', 'EMP5', 'DOC1', 'DOC2', 'DOC3', 'DOC4', 'DOC5', 'SAT1', 'SAT2', 'SAT3', 'SAT4', 'SAT5']

================= KMO & BARTLETT =================
KMO (overall) = 0.916
Bartlett: chi2=1528.702, df=190, p-value=0

KMO per item (sorted):
Item  KMO_item
 RE3  0.824953
SAT2  0.869734
EMP4  0.875136
EMP5  0.891138
EMP3  0.894779
DOC3  0.902249
DOC2  0.905622
DOC1  0.908252
 RE4  0.908827
DOC5  0.910975
 RE5  0.911304
 RE1  0.919772
EMP1  0.933558
SAT5  0.934257
SAT1  0.935145
SAT3  0.936525
DOC4  0.948537
 RE2  0.951082
SAT4  0.952600
EMP2  0.953936

================= EIGENVALUES (Kaiser) =================
 Factor  Eigenvalue
      1     13.6648
      2      1.1662
      3      0.9211
      4      0.7303
      5      0.6488
      6      0.4035
      7      0.3636
      8      0.3468
      9      0.2904
     10      0.2807
     11      0.2172
     12      0.2124
     13      0.1720
     14   

In [68]:
# ============================================================
# REGRESSION (OLS) - 1 KHUNG CODE -> TRẢ KẾT QUẢ NGAY
# - Auto rename: B1/B2/B3/PHẦN C -> RE/EMP/DOC/SAT
# - Composite mean score (RE, EMP, DOC, SAT)
# - OLS multiple regression: SAT ~ RE + EMP + DOC
# - Bảng hệ số + Model fit
# - VIF (đa cộng tuyến)
# - Hồi quy đơn (đối chiếu)
# - Check giả định nhanh (JB normality, Breusch-Pagan)
# ============================================================

import pandas as pd, numpy as np, re
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan

FILE_XLSX = "data_2.xlsx"

# -------------------------
# 0) READ + AUTO RENAME
# -------------------------
df = pd.read_excel(FILE_XLSX, sheet_name=0)
df.columns = [str(c).strip() for c in df.columns]

def pick_cols(pattern: str):
    pat = re.compile(pattern, flags=re.IGNORECASE)
    return [c for c in df.columns if pat.search(str(c))]

b1_cols  = pick_cols(r"\bB1\b")
b2_cols  = pick_cols(r"\bB2\b")
b3_cols  = pick_cols(r"\bB3\b")
sat_cols = pick_cols(r"PHẦN\s*C")

rename_map = {}
for i, c in enumerate(b1_cols,  start=1): rename_map[c] = f"RE{i}"
for i, c in enumerate(b2_cols,  start=1): rename_map[c] = f"EMP{i}"
for i, c in enumerate(b3_cols,  start=1): rename_map[c] = f"DOC{i}"
for i, c in enumerate(sat_cols, start=1): rename_map[c] = f"SAT{i}"

df2 = df.rename(columns=rename_map)
item_cols = list(rename_map.values())

df_items = df2[item_cols].apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)
print("Valid n =", len(df_items))
print("Items:", df_items.columns.tolist())

# -------------------------
# 1) COMPOSITE (MEAN SCORE)
# -------------------------
def scale_mean(prefix):
    cols = [c for c in df_items.columns if str(c).startswith(prefix)]
    return df_items[cols].mean(axis=1)

df_model = pd.DataFrame({
    "RE":  scale_mean("RE"),
    "EMP": scale_mean("EMP"),
    "DOC": scale_mean("DOC"),
    "SAT": scale_mean("SAT"),
})

print("\nComposite head:\n", df_model.head())

# -------------------------
# 2) OLS MULTIPLE REGRESSION: SAT ~ RE + EMP + DOC
# -------------------------
y = df_model["SAT"]
X = df_model[["RE", "EMP", "DOC"]]
X_const = sm.add_constant(X)

model = sm.OLS(y, X_const).fit()

coef_table = pd.DataFrame({
    "coef": model.params,
    "std_err": model.bse,
    "t": model.tvalues,
    "p_value": model.pvalues,
    "CI_low": model.conf_int()[0],
    "CI_high": model.conf_int()[1],
})

fit_table = pd.DataFrame({
    "n": [len(df_model)],
    "R2": [model.rsquared],
    "Adj_R2": [model.rsquared_adj],
    "F_stat": [model.fvalue],
    "F_pvalue": [model.f_pvalue],
    "AIC": [model.aic],
    "BIC": [model.bic],
    "Durbin_Watson": [sm.stats.stattools.durbin_watson(model.resid)],
})

print("\n================= MODEL FIT =================")
print(fit_table.round(6).to_string(index=False))

print("\n================= COEFFICIENTS (OLS) =================")
print(coef_table.round(6).to_string())

print("\n================= FULL SUMMARY (SPSS-like) =================")
print(model.summary())

# -------------------------
# 3) VIF (MULTICOLLINEARITY)
# -------------------------
vif_df = pd.DataFrame({
    "variable": X_const.columns,
    "VIF": [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]
})
print("\n================= VIF =================")
print(vif_df.round(3).to_string(index=False))

# -------------------------
# 4) SIMPLE REGRESSIONS (đối chiếu): SAT ~ từng biến
# -------------------------
simple_rows = []
for var in ["RE", "EMP", "DOC"]:
    X1 = sm.add_constant(df_model[[var]])
    m1 = sm.OLS(y, X1).fit()
    simple_rows.append({
        "model": f"SAT ~ {var}",
        "coef_var": m1.params[var],
        "p_var": m1.pvalues[var],
        "R2": m1.rsquared,
        "Adj_R2": m1.rsquared_adj
    })
simple_df = pd.DataFrame(simple_rows)

print("\n================= SIMPLE OLS (COMPARISON) =================")
print(simple_df.round(6).to_string(index=False))

# -------------------------
# 5) QUICK ASSUMPTION CHECKS (optional)
#    - Normality: Jarque-Bera p > 0.05 => residual ~ normal (tạm ổn)
#    - Homoscedasticity: Breusch-Pagan p > 0.05 => phương sai không đổi (tạm ổn)
# -------------------------
jb_stat, jb_p, skew, kurt = sm.stats.stattools.jarque_bera(model.resid)
bp_stat, bp_p, _, _ = het_breuschpagan(model.resid, X_const)

assump = pd.DataFrame({
    "Jarque_Bera_stat": [jb_stat],
    "Jarque_Bera_p": [jb_p],
    "Skew": [skew],
    "Kurtosis": [kurt],
    "BreuschPagan_stat": [bp_stat],
    "BreuschPagan_p": [bp_p],
})

print("\n================= ASSUMPTION CHECKS =================")
print(assump.round(6).to_string(index=False))


Valid n = 66
Items: ['RE1', 'RE2', 'RE3', 'RE4', 'RE5', 'EMP1', 'EMP2', 'EMP3', 'EMP4', 'EMP5', 'DOC1', 'DOC2', 'DOC3', 'DOC4', 'DOC5', 'SAT1', 'SAT2', 'SAT3', 'SAT4', 'SAT5']

Composite head:
     RE  EMP  DOC  SAT
0  4.0  3.0  4.6  3.2
1  1.0  1.0  1.0  1.0
2  3.0  2.6  4.0  2.8
3  2.6  2.2  1.2  1.6
4  4.4  3.6  3.6  3.4

================= MODEL FIT =================
 n      R2   Adj_R2    F_stat  F_pvalue        AIC        BIC  Durbin_Watson
66 0.79005 0.779891 77.769568       0.0 111.037512 119.796131       2.014783

================= COEFFICIENTS (OLS) =================
           coef   std_err         t   p_value    CI_low   CI_high
const -1.081681  0.323008 -3.348781  0.001385 -1.727364 -0.435999
RE     0.256321  0.165016  1.553309  0.125439 -0.073542  0.586184
EMP    0.508821  0.157847  3.223505  0.002021  0.193289  0.824354
DOC    0.513166  0.131241  3.910090  0.000231  0.250818  0.775514

================= FULL SUMMARY (SPSS-like) =================
                         

In [69]:
# ============================================================
# MEDIATION ANALYSIS (ĐÃ SỬA LỖI dtype object) – Python
# - RE -> EMP -> SAT
# - RE -> DOC -> SAT
# - Parallel: RE -> (EMP, DOC) -> SAT
# + Bootstrap CI cho indirect effects
# ============================================================

import pandas as pd, numpy as np, re
import statsmodels.api as sm

FILE_XLSX = "data_2.xlsx"
BOOT = 5000
SEED = 42
np.random.seed(SEED)

# -------------------------
# 0) READ + AUTO RENAME -> df_items
# -------------------------
df = pd.read_excel(FILE_XLSX, sheet_name=0)
df.columns = [str(c).strip() for c in df.columns]

def pick_cols(pattern: str):
    pat = re.compile(pattern, flags=re.IGNORECASE)
    return [c for c in df.columns if pat.search(str(c))]

b1_cols  = pick_cols(r"\bB1\b")
b2_cols  = pick_cols(r"\bB2\b")
b3_cols  = pick_cols(r"\bB3\b")
sat_cols = pick_cols(r"PHẦN\s*C")

rename_map = {}
for i, c in enumerate(b1_cols,  start=1): rename_map[c] = f"RE{i}"
for i, c in enumerate(b2_cols,  start=1): rename_map[c] = f"EMP{i}"
for i, c in enumerate(b3_cols,  start=1): rename_map[c] = f"DOC{i}"
for i, c in enumerate(sat_cols, start=1): rename_map[c] = f"SAT{i}"

df2 = df.rename(columns=rename_map)
item_cols = list(rename_map.values())

df_items = df2[item_cols].apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)

# -------------------------
# 1) COMPOSITE MEAN -> df_model
# -------------------------
def scale_mean(prefix):
    cols = [c for c in df_items.columns if str(c).startswith(prefix)]
    return df_items[cols].mean(axis=1)

df_model = pd.DataFrame({
    "RE":  scale_mean("RE"),
    "EMP": scale_mean("EMP"),
    "DOC": scale_mean("DOC"),
    "SAT": scale_mean("SAT"),
}).dropna().reset_index(drop=True)

print("n =", len(df_model))

# -------------------------
# 2) Helpers
# -------------------------
def ols_fit(df_in, y, X_list):
    X = sm.add_constant(df_in[X_list])
    return sm.OLS(df_in[y], X).fit()

def pretty_print_dict(d, title):
    """In dict không bị lỗi dtype object; tự round phần số."""
    rows = []
    for k, v in d.items():
        if isinstance(v, (int, float, np.floating)) and not (pd.isna(v)):
            rows.append((k, float(v)))
        else:
            rows.append((k, v))
    out = pd.DataFrame(rows, columns=["metric", "value"])
    # round chỉ với numeric
    out["value_num"] = pd.to_numeric(out["value"], errors="coerce")
    out.loc[out["value_num"].notna(), "value"] = out.loc[out["value_num"].notna(), "value_num"].round(6)
    out = out.drop(columns=["value_num"])
    print("\n" + "="*18 + f" {title} " + "="*18)
    print(out.to_string(index=False))

def mediation_single_boot(df_in, x, m, y, boot=5000):
    # a: m ~ x
    mod_a = ols_fit(df_in, m, [x])
    a = mod_a.params[x]

    # c: y ~ x
    mod_c = ols_fit(df_in, y, [x])
    c = mod_c.params[x]

    # b & c': y ~ x + m
    mod_b = ols_fit(df_in, y, [x, m])
    b = mod_b.params[m]
    c_prime = mod_b.params[x]

    indirect = a * b

    # bootstrap CI for indirect
    n = len(df_in)
    inds = np.empty(boot, dtype=float)
    for i in range(boot):
        samp = df_in.sample(n=n, replace=True)
        a_b = ols_fit(samp, m, [x]).params[x]
        b_b = ols_fit(samp, y, [x, m]).params[m]
        inds[i] = a_b * b_b

    ci_low, ci_high = np.percentile(inds, [2.5, 97.5])
    sig = (ci_low > 0) or (ci_high < 0)

    res = {
        "a (m~x)": a,
        "b (y~x+m, coef m)": b,
        "c total (y~x)": c,
        "c' direct (y~x+m, coef x)": c_prime,
        "indirect a*b": indirect,
        "boot_CI_low_2.5%": ci_low,
        "boot_CI_high_97.5%": ci_high,
        "indirect_sig(CI excludes 0)": sig,
        "p_a(x)": mod_a.pvalues[x],
        "p_b(m)": mod_b.pvalues[m],
        "p_c_total(x)": mod_c.pvalues[x],
        "p_c_prime(x)": mod_b.pvalues[x],
    }
    return res, mod_a, mod_b, mod_c

def mediation_parallel_boot(df_in, x, mediators, y, boot=5000):
    # a paths
    mods_a = {mm: ols_fit(df_in, mm, [x]) for mm in mediators}
    a = {mm: mods_a[mm].params[x] for mm in mediators}

    # y model: y ~ x + mediators
    mod_y = ols_fit(df_in, y, [x] + mediators)
    b = {mm: mod_y.params[mm] for mm in mediators}
    c_prime = mod_y.params[x]

    # total effect
    mod_c = ols_fit(df_in, y, [x])
    c_total = mod_c.params[x]

    indirect = {mm: a[mm] * b[mm] for mm in mediators}
    indirect_total = sum(indirect.values())

    # bootstrap
    n = len(df_in)
    boot_ind = {mm: np.empty(boot, dtype=float) for mm in mediators}
    boot_total = np.empty(boot, dtype=float)

    for i in range(boot):
        samp = df_in.sample(n=n, replace=True)
        a_b = {mm: ols_fit(samp, mm, [x]).params[x] for mm in mediators}
        mod_y_b = ols_fit(samp, y, [x] + mediators)
        b_b = {mm: mod_y_b.params[mm] for mm in mediators}
        ind_b = {mm: a_b[mm] * b_b[mm] for mm in mediators}
        for mm in mediators:
            boot_ind[mm][i] = ind_b[mm]
        boot_total[i] = sum(ind_b.values())

    def ci(arr):
        lo, hi = np.percentile(arr, [2.5, 97.5])
        sig = (lo > 0) or (hi < 0)
        return lo, hi, sig

    rows = []
    for mm in mediators:
        lo, hi, sig = ci(boot_ind[mm])
        rows.append({
            "effect": f"indirect via {mm}",
            "a": a[mm],
            "b": b[mm],
            "a*b": indirect[mm],
            "CI_low": lo,
            "CI_high": hi,
            "sig(CI excludes 0)": sig
        })

    loT, hiT, sigT = ci(boot_total)
    rows += [
        {"effect": "total indirect (sum)", "a": np.nan, "b": np.nan, "a*b": indirect_total, "CI_low": loT, "CI_high": hiT, "sig(CI excludes 0)": sigT},
        {"effect": "direct c' (x in y~x+M)", "a": np.nan, "b": np.nan, "a*b": c_prime, "CI_low": np.nan, "CI_high": np.nan, "sig(CI excludes 0)": np.nan},
        {"effect": "total c (x in y~x)", "a": np.nan, "b": np.nan, "a*b": c_total, "CI_low": np.nan, "CI_high": np.nan, "sig(CI excludes 0)": np.nan},
    ]

    return pd.DataFrame(rows), mods_a, mod_y, mod_c

# -------------------------
# 3) MEDIATION ĐƠN
# -------------------------
res_emp, mod_a_emp, mod_b_emp, mod_c_emp = mediation_single_boot(df_model, "RE", "EMP", "SAT", boot=BOOT)
pretty_print_dict(res_emp, "MEDIATION: RE -> EMP -> SAT")

res_doc, mod_a_doc, mod_b_doc, mod_c_doc = mediation_single_boot(df_model, "RE", "DOC", "SAT", boot=BOOT)
pretty_print_dict(res_doc, "MEDIATION: RE -> DOC -> SAT")

# -------------------------
# 4) PARALLEL MEDIATION
# -------------------------
table_parallel, mods_a, mod_y, mod_c = mediation_parallel_boot(df_model, "RE", ["EMP", "DOC"], "SAT", boot=BOOT)

print("\n" + "="*18 + " PARALLEL MEDIATION: RE -> (EMP, DOC) -> SAT " + "="*18)
# round chỉ các cột số
num_cols = ["a","b","a*b","CI_low","CI_high"]
table_parallel[num_cols] = table_parallel[num_cols].apply(pd.to_numeric, errors="coerce").round(6)
print(table_parallel.to_string(index=False))

# (Tuỳ chọn) nếu muốn xem summary mô hình:
# print(mod_a_emp.summary())  # EMP ~ RE
# print(mod_b_emp.summary())  # SAT ~ RE + EMP
# print(mod_y.summary())      # SAT ~ RE + EMP + DOC


n = 66

================== MEDIATION: RE -> EMP -> SAT ==================
                     metric     value
                    a (m~x)  0.893239
          b (y~x+m, coef m)  0.771184
              c total (y~x)  1.179177
  c' direct (y~x+m, coef x)  0.490325
               indirect a*b  0.688852
           boot_CI_low_2.5%  0.407882
         boot_CI_high_97.5%  1.030577
indirect_sig(CI excludes 0)       1.0
                     p_a(x)       0.0
                     p_b(m)  0.000008
               p_c_total(x)       0.0
               p_c_prime(x)  0.005448

================== MEDIATION: RE -> DOC -> SAT ==================
                     metric     value
                    a (m~x)   0.91268
          b (y~x+m, coef m)  0.693002
              c total (y~x)  1.179177
  c' direct (y~x+m, coef x)  0.546687
               indirect a*b  0.632489
           boot_CI_low_2.5%  0.405377
         boot_CI_high_97.5%  0.908099
indirect_sig(CI excludes 0)       1.0
                     p_